In [2]:
import ast
import os


def extract_functions_from_file(filepath):
    with open(filepath, "r") as f:
        tree = ast.parse(f.read())

    print(f"\nFILE: {os.path.basename(filepath)}")
    print("=" * 80)

    for node in tree.body:  # Changed from ast.walk to tree.body to maintain order
        if isinstance(node, ast.ClassDef):
            # Print class definition
            print(f"\nclass {node.name}:")
            docstring = ast.get_docstring(node) or "No docstring"
            # print(f'    """{docstring}"""')

            # Print methods within the class
            for item in node.body:
                if isinstance(item, ast.FunctionDef):
                    args = [arg.arg for arg in item.args.args]
                    print(f"    def {item.name}({', '.join(args)}):")
            print()

        elif isinstance(node, ast.FunctionDef):
            # Print standalone function
            docstring = ast.get_docstring(node) or "No docstring"
            args = [arg.arg for arg in node.args.args]
            print(f"\ndef {node.name}({', '.join(args)}):")
            # print(f'    """{docstring}"""')
            print()

# Base path
base_path = "/home/mokr/Reservoir_Modelling/ML_Functions"

# Run for each file
files = ["ML_functions.py", "ML_Plots.py", "ML_Models.py", "ML_Metrics.py", "ML_Training.py"]
# 'ML_Plots.py', 'ML_Losses.py', 'ML_Metrics.py',
for filename in files:
    filepath = os.path.join(base_path, filename)
    if os.path.exists(filepath):
        extract_functions_from_file(filepath)
    else:
        print(f"\nWARNING: File not found: {filepath}")


FILE: ML_functions.py

def sample_batch(all_pairs, batch_size, random_seed):


def load_scalers(output_path):


def get_closest_landcover_year(year):


def preprocess_to_tensors(pairs, temporal_data, static_variables, goal):


class TensorCatchmentDataset:
    def __init__(self, tensor_data):
    def __len__(self):
    def __getitem__(self, idx):


class CatchmentDataset:
    def __init__(self, pairs, temporal_data, static_variables, goal):
    def __len__(self):
    def __getitem__(self, idx):


class NSELoss:
    def __init__(self, eps):
    def forward(self, preds, target):


class NNSELoss:
    def __init__(self, eps):
    def forward(self, preds, target):


class KGELoss:
    def __init__(self, reduction):
    def forward(self, simulated, observed):


def calculate_kge(observed, simulated):


def generate_all_catchment_date_pairs(catchments_df, forecast_days, catchment_type, pair_id_col):


def haversine_distance(lat1, lon1, lat2, lon2):


FILE: ML_Plots.py

def display_results_t

In [5]:
import ast
import os
import json
from pathlib import Path
from typing import Dict, List, Set, Tuple
from collections import defaultdict


class FunctionUsageAnalyzer:
    """Analyze function definitions and usage across Python files and notebooks."""

    def __init__(self, root_dir: str, ignore_folders: List[str] = None):
        self.root_dir = Path(root_dir)
        self.ignore_folders = set(ignore_folders) if ignore_folders else set()
        self.function_definitions = defaultdict(list)  # func_name -> [(file, line)]
        self.function_calls = defaultdict(list)  # func_name -> [(file, line)]
        self.class_definitions = defaultdict(list)  # class_name -> [(file, line)]
        self.class_instantiations = defaultdict(list)  # class_name -> [(file, line)]
        self.imports = defaultdict(list)  # file -> [import statements]
        self.file_functions = defaultdict(set)  # file -> set of defined functions
        self.file_classes = defaultdict(set)  # file -> set of defined classes
        self.class_methods = defaultdict(list)  # class_name -> [method_names]

    def _should_ignore(self, filepath: Path) -> bool:
        """Check if a file should be ignored based on ignore_folders."""
        try:
            rel_path = filepath.relative_to(self.root_dir)
            # Check if any part of the path matches an ignored folder
            for part in rel_path.parts:
                if part in self.ignore_folders:
                    return True
            # Also check full relative path for nested folders
            rel_path_str = str(rel_path)
            for ignored in self.ignore_folders:
                if ignored in rel_path_str:
                    return True
            return False
        except ValueError:
            return False

    def analyze_directory(self):
        """Analyze all Python files and notebooks in the directory."""
        print(f"Analyzing directory: {self.root_dir}")
        if self.ignore_folders:
            print(f"Ignoring folders: {', '.join(sorted(self.ignore_folders))}")
        print("=" * 80)

        # Find all Python files and notebooks
        all_py_files = list(self.root_dir.rglob("*.py"))
        all_ipynb_files = list(self.root_dir.rglob("*.ipynb"))

        # Filter out ignored folders
        py_files = [f for f in all_py_files if not self._should_ignore(f)]
        ipynb_files = [f for f in all_ipynb_files if not self._should_ignore(f)]

        ignored_py = len(all_py_files) - len(py_files)
        ignored_nb = len(all_ipynb_files) - len(ipynb_files)

        print(f"\nFound {len(py_files)} Python files and {len(ipynb_files)} notebooks")
        if ignored_py or ignored_nb:
            print(f"Ignored {ignored_py} Python files and {ignored_nb} notebooks")
        print()

        # Analyze Python files
        for filepath in py_files:
            self._analyze_python_file(filepath)

        # Analyze notebooks
        for filepath in ipynb_files:
            self._analyze_notebook(filepath)

    def _analyze_python_file(self, filepath: Path):
        """Analyze a single Python file."""
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                content = f.read()

            tree = ast.parse(content, filename=str(filepath))
            rel_path = filepath.relative_to(self.root_dir)

            # Extract imports
            self._extract_imports(tree, str(rel_path))

            # Extract function definitions and calls
            self._extract_functions_and_calls(tree, str(rel_path))

            print(f"✓ Analyzed: {rel_path}")

        except Exception as e:
            print(f"✗ Error analyzing {filepath.name}: {e}")

    def _analyze_notebook(self, filepath: Path):
        """Analyze a Jupyter notebook."""
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                notebook = json.load(f)

            rel_path = filepath.relative_to(self.root_dir)

            # Combine all code cells
            code_cells = [
                cell["source"]
                for cell in notebook.get("cells", [])
                if cell.get("cell_type") == "code"
            ]

            for cell_idx, source in enumerate(code_cells):
                if isinstance(source, list):
                    source = "".join(source)

                try:
                    tree = ast.parse(source)

                    # Extract imports
                    self._extract_imports(tree, f"{rel_path}#cell{cell_idx}")

                    # Extract function definitions and calls
                    self._extract_functions_and_calls(
                        tree, f"{rel_path}#cell{cell_idx}"
                    )

                except SyntaxError:
                    pass  # Skip cells with syntax errors (incomplete code, etc.)

            print(f"✓ Analyzed: {rel_path}")

        except Exception as e:
            print(f"✗ Error analyzing {filepath.name}: {e}")

    def _extract_imports(self, tree: ast.AST, file_id: str):
        """Extract import statements from AST."""
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                for alias in node.names:
                    import_str = f"import {alias.name}"
                    if alias.asname:
                        import_str += f" as {alias.asname}"
                    self.imports[file_id].append(import_str)

            elif isinstance(node, ast.ImportFrom):
                module = node.module or ""
                for alias in node.names:
                    import_str = f"from {module} import {alias.name}"
                    if alias.asname:
                        import_str += f" as {alias.asname}"
                    self.imports[file_id].append(import_str)

    def _extract_functions_and_calls(self, tree: ast.AST, file_id: str):
        """Extract function definitions, calls, and class definitions from AST."""
        for node in ast.walk(tree):
            # Class definitions
            if isinstance(node, ast.ClassDef):
                class_name = node.name
                line = getattr(node, "lineno", 0)
                self.class_definitions[class_name].append((file_id, line))
                self.file_classes[file_id].add(class_name)

                # Extract methods from the class
                methods = []
                for item in node.body:
                    if isinstance(item, ast.FunctionDef):
                        methods.append(item.name)
                        # Also record methods as functions
                        func_name = item.name
                        func_line = getattr(item, "lineno", 0)
                        self.function_definitions[func_name].append(
                            (f"{file_id}::{class_name}", func_line)
                        )

                self.class_methods[class_name] = methods

            # Function definitions (top-level only, methods handled above)
            elif isinstance(node, ast.FunctionDef):
                func_name = node.name
                line = getattr(node, "lineno", 0)
                # Check if this is not inside a class (top-level function)
                parent_is_class = False
                for parent in ast.walk(tree):
                    if isinstance(parent, ast.ClassDef):
                        if node in ast.walk(parent):
                            parent_is_class = True
                            break

                if not parent_is_class:
                    self.function_definitions[func_name].append((file_id, line))
                    self.file_functions[file_id].add(func_name)

            # Function/method calls
            elif isinstance(node, ast.Call):
                func_name = self._get_call_name(node.func)
                if func_name:
                    line = getattr(node, "lineno", 0)
                    self.function_calls[func_name].append((file_id, line))

                # Check if this is a class instantiation
                if isinstance(node.func, ast.Name):
                    # Heuristic: if it looks like a class name (starts with capital)
                    class_name = node.func.id
                    if class_name and class_name[0].isupper():
                        self.class_instantiations[class_name].append((file_id, line))

    def _get_call_name(self, node) -> str:
        """Extract function name from a call node."""
        if isinstance(node, ast.Name):
            return node.id
        elif isinstance(node, ast.Attribute):
            # For method calls like obj.method(), we just get 'method'
            return node.attr
        return ""

    def find_duplicate_function_names(self) -> Dict[str, List[Tuple[str, int]]]:
        """Find functions defined in multiple places."""
        duplicates = {}
        for func_name, locations in self.function_definitions.items():
            if len(locations) > 1:
                duplicates[func_name] = locations
        return duplicates

    def find_duplicate_class_names(self) -> Dict[str, List[Tuple[str, int]]]:
        """Find classes defined in multiple places."""
        duplicates = {}
        for class_name, locations in self.class_definitions.items():
            if len(locations) > 1:
                duplicates[class_name] = locations
        return duplicates

    def find_unused_functions(self) -> Dict[str, List[Tuple[str, int]]]:
        """Find functions that are defined but never called."""
        unused = {}
        for func_name, definitions in self.function_definitions.items():
            # Skip magic methods and private methods (might be used elsewhere)
            if func_name.startswith("_"):
                continue

            # Check if function is called
            calls = self.function_calls.get(func_name, [])

            # Filter out self-references (function calling itself or defined in same file)
            external_calls = []
            for call_file, call_line in calls:
                # Remove cell info from notebooks for comparison
                call_file_base = call_file.split("#")[0]

                # Check if call is from a different file
                is_external = True
                for def_file, def_line in definitions:
                    def_file_base = def_file.split("#")[0].split("::")[0]
                    if call_file_base == def_file_base:
                        is_external = False
                        break

                if is_external:
                    external_calls.append((call_file, call_line))

            if not external_calls:
                unused[func_name] = definitions

        return unused

    def find_unused_classes(self) -> Dict[str, List[Tuple[str, int]]]:
        """Find classes that are defined but never instantiated."""
        unused = {}
        for class_name, definitions in self.class_definitions.items():
            # Skip private classes
            if class_name.startswith("_"):
                continue

            # Check if class is instantiated
            instantiations = self.class_instantiations.get(class_name, [])

            # Filter out self-references
            external_instantiations = []
            for inst_file, inst_line in instantiations:
                inst_file_base = inst_file.split("#")[0]

                is_external = True
                for def_file, def_line in definitions:
                    def_file_base = def_file.split("#")[0]
                    if inst_file_base == def_file_base:
                        is_external = False
                        break

                if is_external:
                    external_instantiations.append((inst_file, inst_line))

            if not external_instantiations:
                unused[class_name] = definitions

        return unused

    def generate_report(self, output_file: str = "function_analysis_report.txt"):
        """Generate a comprehensive report."""
        with open(output_file, "w", encoding="utf-8") as f:
            f.write("PYTHON PROJECT FUNCTION USAGE ANALYSIS\n")
            f.write("=" * 80 + "\n\n")

            # Summary
            f.write("SUMMARY:\n")
            f.write("-" * 80 + "\n")
            f.write(f"Total files analyzed: {len(self.file_functions)}\n")
            f.write(
                f"Total unique functions defined: {len(self.function_definitions)}\n"
            )
            f.write(f"Total unique classes defined: {len(self.class_definitions)}\n")
            f.write(f"Total unique functions called: {len(self.function_calls)}\n")
            f.write(
                f"Total class instantiations tracked: {len(self.class_instantiations)}\n\n"
            )

            # Duplicate class names
            class_duplicates = self.find_duplicate_class_names()
            f.write(f"\nDUPLICATE CLASS NAMES ({len(class_duplicates)}):\n")
            f.write("=" * 80 + "\n")
            if class_duplicates:
                for class_name, locations in sorted(class_duplicates.items()):
                    f.write(f"\n'{class_name}' defined in {len(locations)} places:\n")
                    for file_id, line in locations:
                        f.write(f"  - {file_id}:{line}\n")

                    # Show methods
                    methods = self.class_methods.get(class_name, [])
                    if methods:
                        f.write(f"  Methods: {', '.join(methods)}\n")

                    # Show where it's instantiated
                    instantiations = self.class_instantiations.get(class_name, [])
                    if instantiations:
                        f.write(f"  Instantiated in {len(instantiations)} place(s):\n")
                        for inst_file, inst_line in instantiations[:5]:
                            f.write(f"    • {inst_file}:{inst_line}\n")
                        if len(instantiations) > 5:
                            f.write(f"    ... and {len(instantiations) - 5} more\n")
                    else:
                        f.write("  ⚠️  Not instantiated anywhere!\n")
            else:
                f.write("No duplicate class names found.\n")

            # Unused classes
            unused_classes = self.find_unused_classes()
            f.write(f"\n\nPOTENTIALLY UNUSED CLASSES ({len(unused_classes)}):\n")
            f.write("=" * 80 + "\n")
            f.write("(Classes defined but not instantiated from other files)\n\n")
            if unused_classes:
                for class_name, locations in sorted(unused_classes.items()):
                    f.write(f"'{class_name}':\n")
                    for file_id, line in locations:
                        f.write(f"  - {file_id}:{line}\n")
                    methods = self.class_methods.get(class_name, [])
                    if methods:
                        f.write(f"  Methods: {', '.join(methods)}\n")
            else:
                f.write("All classes appear to be used.\n")

            # Duplicate function names
            duplicates = self.find_duplicate_function_names()
            f.write(f"\nDUPLICATE FUNCTION NAMES ({len(duplicates)}):\n")
            f.write("=" * 80 + "\n")
            if duplicates:
                for func_name, locations in sorted(duplicates.items()):
                    f.write(f"\n'{func_name}' defined in {len(locations)} places:\n")
                    for file_id, line in locations:
                        f.write(f"  - {file_id}:{line}\n")

                    # Show where it's called
                    calls = self.function_calls.get(func_name, [])
                    if calls:
                        f.write(f"  Called in {len(calls)} place(s):\n")
                        for call_file, call_line in calls[:5]:  # Show first 5
                            f.write(f"    • {call_file}:{call_line}\n")
                        if len(calls) > 5:
                            f.write(f"    ... and {len(calls) - 5} more\n")
                    else:
                        f.write("  ⚠️  Not called anywhere!\n")
            else:
                f.write("No duplicate function names found.\n")

            # Unused functions
            unused = self.find_unused_functions()
            f.write(f"\n\nPOTENTIALLY UNUSED FUNCTIONS ({len(unused)}):\n")
            f.write("=" * 80 + "\n")
            f.write("(Functions defined but not called from other files)\n\n")
            if unused:
                for func_name, locations in sorted(unused.items()):
                    f.write(f"'{func_name}':\n")
                    for file_id, line in locations:
                        f.write(f"  - {file_id}:{line}\n")
            else:
                f.write("All functions appear to be used.\n")

            # Functions and Classes per file
            f.write(f"\n\nFUNCTIONS AND CLASSES DEFINED PER FILE:\n")
            f.write("=" * 80 + "\n")
            for file_id in sorted(
                set(list(self.file_functions.keys()) + list(self.file_classes.keys()))
            ):
                funcs = sorted(self.file_functions.get(file_id, set()))
                classes = sorted(self.file_classes.get(file_id, set()))

                f.write(f"\n{file_id}:\n")

                if classes:
                    f.write(f"  Classes ({len(classes)}):\n")
                    for cls in classes:
                        methods = self.class_methods.get(cls, [])
                        f.write(f"    - {cls}")
                        if methods:
                            f.write(f" (methods: {', '.join(methods)})")
                        f.write("\n")

                if funcs:
                    f.write(f"  Functions ({len(funcs)}):\n")
                    for func in funcs:
                        f.write(f"    - {func}\n")

            # Imports per file
            f.write(f"\n\nIMPORTS PER FILE:\n")
            f.write("=" * 80 + "\n")
            for file_id in sorted(self.imports.keys()):
                imports = self.imports[file_id]
                f.write(f"\n{file_id} ({len(imports)} imports):\n")
                for imp in sorted(set(imports)):
                    f.write(f"  - {imp}\n")

            # Class instantiation details
            f.write(f"\n\nCLASS INSTANTIATION DETAILS:\n")
            f.write("=" * 80 + "\n")
            for class_name in sorted(self.class_definitions.keys()):
                definitions = self.class_definitions[class_name]
                instantiations = self.class_instantiations.get(class_name, [])

                f.write(f"\n'{class_name}':\n")
                f.write(f"  Defined in:\n")
                for def_file, def_line in definitions:
                    f.write(f"    - {def_file}:{def_line}\n")

                methods = self.class_methods.get(class_name, [])
                if methods:
                    f.write(f"  Methods: {', '.join(methods)}\n")

                if instantiations:
                    f.write(f"  Instantiated from ({len(instantiations)} locations):\n")
                    for inst_file, inst_line in instantiations[:10]:
                        f.write(f"    - {inst_file}:{inst_line}\n")
                    if len(instantiations) > 10:
                        f.write(f"    ... and {len(instantiations) - 10} more\n")
                else:
                    f.write(f"  ⚠️  Not instantiated (or instantiation not detected)\n")

            # Function call matrix
            f.write(f"\n\nFUNCTION CALL DETAILS:\n")
            f.write("=" * 80 + "\n")
            for func_name in sorted(self.function_calls.keys()):
                definitions = self.function_definitions.get(func_name, [])
                calls = self.function_calls[func_name]

                f.write(f"\n'{func_name}':\n")
                if definitions:
                    f.write(f"  Defined in:\n")
                    for def_file, def_line in definitions:
                        f.write(f"    - {def_file}:{def_line}\n")
                else:
                    f.write(f"  ⚠️  Not defined in this project (external/builtin)\n")

                f.write(f"  Called from ({len(calls)} locations):\n")
                for call_file, call_line in calls[:10]:  # Show first 10
                    f.write(f"    - {call_file}:{call_line}\n")
                if len(calls) > 10:
                    f.write(f"    ... and {len(calls) - 10} more\n")

        print(f"\n{'=' * 80}")
        print(f"✓ Report saved to: {output_file}")
        print(f"{'=' * 80}\n")

    def print_summary(self):
        """Print a quick summary to console."""
        duplicates = self.find_duplicate_function_names()
        unused = self.find_unused_functions()
        class_duplicates = self.find_duplicate_class_names()
        unused_classes = self.find_unused_classes()

        print("\n" + "=" * 80)
        print("ANALYSIS SUMMARY")
        print("=" * 80)
        print(f"Files analyzed: {len(self.file_functions)}")
        print(f"Unique functions defined: {len(self.function_definitions)}")
        print(f"Unique classes defined: {len(self.class_definitions)}")
        print(f"Duplicate function names: {len(duplicates)}")
        print(f"Duplicate class names: {len(class_duplicates)}")
        print(f"Potentially unused functions: {len(unused)}")
        print(f"Potentially unused classes: {len(unused_classes)}")

        if class_duplicates:
            print(f"\n⚠️  DUPLICATE CLASS NAMES:")
            for class_name, locations in sorted(class_duplicates.items())[:10]:
                print(f"  • {class_name}: {len(locations)} definitions")
            if len(class_duplicates) > 10:
                print(f"  ... and {len(class_duplicates) - 10} more")

        if duplicates:
            print(f"\n⚠️  DUPLICATE FUNCTION NAMES:")
            for func_name, locations in sorted(duplicates.items())[:10]:
                print(f"  • {func_name}: {len(locations)} definitions")
            if len(duplicates) > 10:
                print(f"  ... and {len(duplicates) - 10} more")

        if unused_classes:
            print(f"\n⚠️  POTENTIALLY UNUSED CLASSES (sample):")
            for class_name in sorted(unused_classes.keys())[:10]:
                print(f"  • {class_name}")
            if len(unused_classes) > 10:
                print(f"  ... and {len(unused_classes) - 10} more")

        if unused:
            print(f"\n⚠️  POTENTIALLY UNUSED FUNCTIONS (sample):")
            for func_name in sorted(unused.keys())[:10]:
                print(f"  • {func_name}")
            if len(unused) > 10:
                print(f"  ... and {len(unused) - 10} more")

        print("\n" + "=" * 80 + "\n")


# Example usage
if __name__ == "__main__":
    # Set your project directory here
    project_dir = "/home/mokr/Reservoir_Modelling"

    # Folders to ignore (can use simple names or relative paths)
    ignore_folders = [
        "Old_Code",
        "mlruns",
        "__pycache__",
        ".git",
        ".ipynb_checkpoints",
        "Sample_Pairs",
        "results",
        "CamelsGB_Models",
        "Catchment_Lists",
        "Old_Functions",
        "Ensemble_Code",
        
    ]

    # Create analyzer
    analyzer = FunctionUsageAnalyzer(project_dir, ignore_folders=ignore_folders)

    # Analyze all files
    analyzer.analyze_directory()

    # Print summary
    analyzer.print_summary()

    # Generate detailed report
    analyzer.generate_report("function_analysis_report.txt")

    # Optional: Get specific information
    print("\nDuplicate classes that need attention:")
    class_duplicates = analyzer.find_duplicate_class_names()
    for class_name, locations in sorted(class_duplicates.items()):
        print(f"\n{class_name}:")
        for file_id, line in locations:
            print(f"  - {file_id}:{line}")

    print("\nDuplicate functions that need attention:")
    duplicates = analyzer.find_duplicate_function_names()
    for func_name, locations in sorted(duplicates.items()):
        print(f"\n{func_name}:")
        for file_id, line in locations:
            print(f"  - {file_id}:{line}")

Analyzing directory: /home/mokr/Reservoir_Modelling
Ignoring folders: .git, .ipynb_checkpoints, CamelsGB_Models, Catchment_Lists, Ensemble_Code, Old_Code, Old_Functions, Sample_Pairs, __pycache__, mlruns, results

Found 14 Python files and 7 notebooks
Ignored 52 Python files and 13 notebooks

✓ Analyzed: ML_Functions/ML_Models.py
✓ Analyzed: ML_Functions/ML_Metrics.py
✓ Analyzed: ML_Functions/ML_functions.py
✓ Analyzed: ML_Functions/ML_Training.py
✓ Analyzed: ML_Functions/ML_Plots.py
✓ Analyzed: UK_Batch_Scripts/Making_Tensors.py
✓ Analyzed: UK_Batch_Scripts/Full_Hyperparameter_Training.py
✓ Analyzed: UK_Batch_Scripts/Training_LSTM.py
✓ Analyzed: UK_Batch_Scripts/Reservoir_Hyperparameter_Training.py
✓ Analyzed: UK_Batch_Scripts/Natural_Hyperparameter_Training.py
✓ Analyzed: UK_Batch_Scripts/Hydra_Hyperparameter_Training.py
✓ Analyzed: UK_Batch_Scripts/Getting_tensor_metadata.py
✓ Analyzed: UK_Batch_Scripts/Evaluation_Script.py
✓ Analyzed: UK_Batch_Scripts/mlflow_Reservoir_Hyperparamete